In [27]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from mpl_toolkits.axes_grid1 import make_axes_locatable
import os
from matplotlib.colors import LinearSegmentedColormap

# 颜色
c = 'coral'
c1 = 'yellowgreen'
base_c = 'aqua'

# 坐标轴刻度大小（和Beta_Collocation_points_plots.ipynb一致）
ticks_fontsize = 10

# XY轴标签字体大小（和Beta_Collocation_points_plots.ipynb一致）
label_fontsize = 14

# 动态colorbar格式化函数
def format_colorbar_tick(x, p):
    """
    动态设置colorbar刻度的小数点位数
    绝对值<1: 3位; 1<=绝对值<10: 2位; 10<=绝对值<100: 1位; 绝对值>=100: 0位
    """
    abs_x = abs(x)
    if abs_x < 1:
        return f'{x:.3f}'
    elif abs_x < 10:
        return f'{x:.2f}'
    elif abs_x < 100:
        return f'{x:.1f}'
    else:
        return f'{x:.0f}'

def truncated_cmap(cmap_name, minval=0.0, maxval=1.0, n=256):
    """Return a copy of cmap_name trimmed to [minval, maxval] to soften extremes."""
    cmap = plt.get_cmap(cmap_name)
    colors = cmap(np.linspace(minval, maxval, n))
    return LinearSegmentedColormap.from_list(f"{cmap_name}_trunc", colors)

In [28]:
REGIME_CURVES_FRAC_MAP = {
    "default": [
        [(0.67, 1.0), (0.67, 0.5)],
        [(0.67, 0.5), (1.0, 0.3)],
        [(0.67, 0.5), (0.2, 0.0)],
    ],
}


In [29]:

REGIME_LINE_STYLE = {"color": "black", "linewidth": 2.0, "alpha": 0.9}

def _interp_index(fraction, length):
    """Convert fractional coordinate to grid index based on axis length."""
    if length <= 1:
        return 0.0
    return fraction * (length - 1)

def build_regime_lines(shape, func="default"):
    """
    Turn fractional regime curves into actual grid coordinates for plotting.

    Args:
        shape: (rows, cols) of the heatmap array.
        func: key in REGIME_CURVES_FRAC_MAP to select curve config.
    """
    if not shape or len(shape) != 2:
        return []
    n_rows, n_cols = shape
    curves_frac = REGIME_CURVES_FRAC_MAP.get(func, REGIME_CURVES_FRAC_MAP.get("default", []))
    lines = []
    for curve in curves_frac:
        xs, ys = [], []
        for fx, fy in curve:
            xs.append(_interp_index(fx, n_cols))
            ys.append(_interp_index(fy, n_rows))
        if xs and ys:
            lines.append((xs, ys))
    return lines


In [30]:
import sys, pathlib, importlib

REPO_ROOT = pathlib.Path('/jumbo/yaoqingyang/yuanzhehu/neuraloperators-TL-scaling')
BOUNDARY_UTILS_DIR = REPO_ROOT / 'ipynb/2d_plots'
if str(BOUNDARY_UTILS_DIR) not in sys.path:
    sys.path.append(str(BOUNDARY_UTILS_DIR))

import plot_boundary_utils as boundary_utils
beta_plots = importlib.import_module('Beta_Collocation_points_plots_v4')

plot_2d_phase = beta_plots.plot_2d_phase
merge_overlays = beta_plots.merge_overlays
build_boundary_overlays = boundary_utils.build_boundary_overlays
build_regime_overlay_from_shape = beta_plots.build_regime_overlay_from_shape
get_regime_positions = beta_plots.get_regime_positions

# Poisson-specific tweaks for regime label placement and boundary smoothing
beta_plots.REGIME_POSITION_FRAC_MAP['ad_scale'] = {
    1: (0.18, 0.65),
    2: (0.75, 0.65),
    3: (0.45, 0.19),
}

beta_plots.BOUNDARY_CONFIG_PRESETS['ad_scale'] = {
    'smoothing_sigma': 0.1,
    'threshold_ratio': 0.5,
    'threshold_ratio_map': {'training_loss': 0.7, 'test_error': 0.15},
    'fuse_threshold':10.0
}

### sys-1

### ad-scale experiments (results_sameiteration)

In [31]:
import re
import numpy as np

ad_scale_keys = [
    'ad_scale_0p2_0p4_val1024_1M',
    'ad_scale_0p4_0p6_val1024_1M',
    'ad_scale_0p6_0p8_val1024_1M',
    'ad_scale_0p8_1p0_val1024_1M',
    'ad_scale_1p0_1p3_val1024_1M',
    'ad_scale_1p3_1p7_val1024_1M',
    'ad_scale_1p7_2p0_val1024_1M',
    'ad_scale_2p0_2p5_val1024_1M',
]

ad_scale_labels = [
    '0.2-', '0.4-', '0.6-', '0.8-',
    '1.0-', '1.3-', '1.7-', '-2.5',
]

lr_range = [0.001] 
sub_sampling_range = [256, 128, 64, 32, 16, 8 ,4, 2]
data_number_range = ['128','256', '512', '1024', '2048', '4096', '8192', '16384']
seed_range = [2021, 2022, 2023, 2024, 2025]

bsz = 128
exp_type = 'train'
results_root = '/jumbo/yaoqingyang/yuanzhehu/neuraloperators-TL-scaling/results_sameiteration'

# subsample -> base directory and stop epoch settings
base_dir_by_subsample = {
    256: 'expts_eps1000',
    128: 'expts_eps750',
    64: 'expts_eps500',
    32: 'expts_eps300',
    16: 'expts_eps200',
    8: 'expts_eps150',
    4: 'expts_eps100',
    2: 'expts_eps75',
}
stop_epoch_by_subsample = {
    256: 999,
    128: 749,
    64: 499,
    32: 299,
    16: 199,
    8: 149,
    4: 99,
    2: 74,
}


In [32]:
exp_stats = {}

for scale_key in ad_scale_keys:
    exp_stats[scale_key] = {}
    for subsamples in sub_sampling_range:
        for lr in lr_range:
            hyper_settings = f'bsz{bsz}_lr{lr}_subsample{subsamples}'
            exp_stats[scale_key][hyper_settings] = {}
            best_test_err_by_seed = []
            best_train_err_by_seed = []

            base_dir = f"{results_root}/{base_dir_by_subsample[subsamples]}"
            stop_epoch = stop_epoch_by_subsample.get(subsamples, 999)

            for seed in seed_range:
                hyper_settings_tmp = f'{hyper_settings}/seed{seed}'
                best_log = f'{base_dir}/{scale_key}/{exp_type}/{hyper_settings_tmp}/logs.txt'

                test_err = 1e999
                train_err = 1e999
                try:
                    with open(best_log, 'r') as f:
                        lines = f.readlines()
                        for line in lines:
                            if 'best_val_err' in line:
                                test_err = float(re.search(r"tensor\(\[([0-9.]+)\]", line).group(1))
                            elif 'tr_err' in line:
                                train_err = float(re.search(r"tensor\(\[([0-9.]+)\]", line).group(1))

                            if f'epoch,{stop_epoch}' in line:
                                break

                    best_test_err_by_seed.append(test_err)
                    best_train_err_by_seed.append(train_err)
                except Exception as e:
                    print(f'Error reading {best_log}: {e}')

            exp_stats[scale_key][hyper_settings]['test_err_mean'] = np.mean(best_test_err_by_seed)
            exp_stats[scale_key][hyper_settings]['test_err_std'] = np.std(best_test_err_by_seed)
            exp_stats[scale_key][hyper_settings]['train_err_mean'] = np.mean(best_train_err_by_seed)
            exp_stats[scale_key][hyper_settings]['train_err_std'] = np.std(best_train_err_by_seed)


In [33]:
# Build ad-scale axes and train/test grids for downstream plots
ad_scale_ranges = [
    (0.2, 0.4),
    (0.4, 0.6),
    (0.6, 0.8),
    (0.8, 1.0),
    (1.0, 1.3),
    (1.3, 1.7),
    (1.7, 2.0),
    (2.0, 2.5),
]

# Use labels as x-axis ticks (kept name for compatibility with later cells)
k_range_list = ad_scale_labels

pde_settings = ad_scale_labels
lr = lr_range[0]

# 8x8 grids (subsampling x ad-scale)
test_loss_grid = np.zeros((len(sub_sampling_range), len(pde_settings)))
train_loss_grid = np.zeros_like(test_loss_grid)

for scale_key in ad_scale_keys:
    for subsamples in sub_sampling_range:
        if subsamples == 512:
            hyper_setting = f'bsz64_lr{lr}_subsample{subsamples}'
        elif subsamples == 1024:
            hyper_setting = f'bsz32_lr{lr}_subsample{subsamples}'
        elif subsamples == 2048:
            hyper_setting = f'bsz16_lr{lr}_subsample{subsamples}'
        elif subsamples == 4096:
            hyper_setting = f'bsz8_lr{lr}_subsample{subsamples}'
        else:
            hyper_setting = f'bsz{bsz}_lr{lr}_subsample{subsamples}'

        i = sub_sampling_range.index(subsamples)
        j = ad_scale_keys.index(scale_key)
        test_loss_grid[i, j] = exp_stats[scale_key][hyper_setting]['test_err_mean']
        train_loss_grid[i, j] = exp_stats[scale_key][hyper_setting]['train_err_mean']

print(f"Built ad-scale grids: train {train_loss_grid.shape}, test {test_loss_grid.shape}")


Built ad-scale grids: train (8, 8), test (8, 8)


### CKA / Hessian / LMC


In [34]:
hessian_dir ="/jumbo/yaoqingyang/yuanzhehu/neuraloperators-TL-scaling/hessian_analysis/hessian_analysis_old"


### Hessian Analysis - Sharpness Trace and Eigenvalue

In [35]:
import os
import re
import numpy as np
from pathlib import Path

# Read Hessian sharpness data for ad-scale experiments (skip missing files quietly)
hessian_stats = {}
missing_files = 0

for scale_key, k_range in zip(ad_scale_keys, ad_scale_ranges):
    pde_settings = scale_key
    hessian_stats[pde_settings] = {}
    
    for subsamples in sub_sampling_range:
        for lr in lr_range:
            if subsamples == 256:
                hessian_base_dir = f'{hessian_dir}/expts_eps1000'
                bsz_tmp = 128
            elif subsamples == 128:
                hessian_base_dir = f'{hessian_dir}/expts_eps750'
                bsz_tmp = 128
            elif subsamples == 64:
                hessian_base_dir = f'{hessian_dir}/expts_eps500'
                bsz_tmp = 128
            elif subsamples == 32:
                hessian_base_dir = f'{hessian_dir}/expts_eps300'
                bsz_tmp = 128
            elif subsamples == 16:
                hessian_base_dir = f'{hessian_dir}/expts_eps200'
                bsz_tmp = 128
            elif subsamples == 8:
                hessian_base_dir = f'{hessian_dir}/expts_eps150'
                bsz_tmp = 128
            elif subsamples == 4:
                hessian_base_dir = f'{hessian_dir}/expts_eps100'
                bsz_tmp = 128
            elif subsamples == 2:
                hessian_base_dir = f'{hessian_dir}/expts_eps75'
                bsz_tmp = 128
            elif subsamples == 512:
                hessian_base_dir = f'{hessian_dir}/expts_eps1000'
                bsz_tmp = 64
            elif subsamples == 1024:
                hessian_base_dir = f'{hessian_dir}/expts_eps1000'
                bsz_tmp = 32
            elif subsamples == 2048:
                hessian_base_dir = f'{hessian_dir}/expts_eps1000'
                bsz_tmp = 16
            elif subsamples == 4096:
                hessian_base_dir = f'{hessian_dir}/expts_eps1000'
                bsz_tmp = 8
            else:
                hessian_base_dir = f'{hessian_dir}/expts_eps1000'
                bsz_tmp = 128

            hyper_settings = f'bsz{bsz_tmp}_lr{lr}_subsample{subsamples}'
            hessian_stats[pde_settings][hyper_settings] = {}
            sharpness_trace_by_seed = []
            sharpness_eigenval_by_seed = []
            
            # average over seeds
            for seed in seed_range:
                hessian_file = Path(hessian_base_dir) / pde_settings / 'hessian_analysis' / hyper_settings / f'seed{seed}' / 'hessian_sharpness_results.txt'
                if not hessian_file.exists():
                    missing_files += 1
                    continue
                try:
                    with hessian_file.open('r') as f:
                        for line in f:
                            if line.startswith('sharpness_trace:'):
                                sharpness_trace = float(line.split(':')[1].strip())
                                sharpness_trace_by_seed.append(sharpness_trace)
                            elif line.startswith('sharpness_eigenval:'):
                                sharpness_eigenval = float(line.split(':')[1].strip())
                                sharpness_eigenval_by_seed.append(sharpness_eigenval)
                except Exception as e:
                    missing_files += 1
                    continue
            
            hessian_stats[pde_settings][hyper_settings]['sharpness_trace_mean'] = np.mean(sharpness_trace_by_seed) if len(sharpness_trace_by_seed) > 0 else np.nan
            hessian_stats[pde_settings][hyper_settings]['sharpness_trace_std'] = np.std(sharpness_trace_by_seed) if len(sharpness_trace_by_seed) > 0 else np.nan
            hessian_stats[pde_settings][hyper_settings]['sharpness_eigenval_mean'] = np.mean(sharpness_eigenval_by_seed) if len(sharpness_eigenval_by_seed) > 0 else np.nan
            hessian_stats[pde_settings][hyper_settings]['sharpness_eigenval_std'] = np.std(sharpness_eigenval_by_seed) if len(sharpness_eigenval_by_seed) > 0 else np.nan

print(f"Hessian data loaded (missing files skipped: {missing_files}).")


Hessian data loaded (missing files skipped: 320).


In [36]:
import numpy as np

pde_settings = ad_scale_labels
lr = lr_range[0]

sharpness_eigenval_grid = np.zeros((len(sub_sampling_range), len(pde_settings)))
print(f"Sharpness eigenval grid shape: {sharpness_eigenval_grid.shape}")

for scale_key in ad_scale_keys:
    for subsamples in sub_sampling_range:
        if subsamples == 512:
            hyper_setting = f'bsz64_lr{lr}_subsample{subsamples}'
        elif subsamples == 1024:
            hyper_setting = f'bsz32_lr{lr}_subsample{subsamples}'
        elif subsamples == 2048:
            hyper_setting = f'bsz16_lr{lr}_subsample{subsamples}'
        elif subsamples == 4096:
            hyper_setting = f'bsz8_lr{lr}_subsample{subsamples}'
        else:
            hyper_setting = f'bsz{bsz}_lr{lr}_subsample{subsamples}'

        eigenval = hessian_stats.get(scale_key, {}).get(hyper_setting, {}).get('sharpness_eigenval_mean', np.nan)
        sharpness_eigenval_grid[sub_sampling_range.index(subsamples), ad_scale_keys.index(scale_key)] = (
            np.log10(eigenval) if eigenval is not None and eigenval > 0 else np.nan
        )

print('Hessian grid built (log10).')


Sharpness eigenval grid shape: (8, 8)
Hessian grid built (log10).


In [39]:
import numpy as np

func = 'ad_scale'
x_axis = ad_scale_labels
y_axis = data_number_range

# Collect metrics for the shared plotting util
plot_data = {
    'training_loss': train_loss_grid,
    'test_error': test_loss_grid,
}

# Data-driven fused boundary thresholds; tweak buffers if you want looser/stricter masking
median_train = 0.066  # float(np.nanmedian(train_loss_grid))
median_test = 0.15  # float(np.nanmedian(test_loss_grid))
beta_plots.FUSED_BOUNDARY_PARAMS[func] = {
    **beta_plots.FUSED_BOUNDARY_DEFAULT,
    'train_thresh': median_train,
    'test_thresh': median_test,
    'train_buffer': 0.01,
    'test_buffer': 0.15,
    'smooth_sigma': 0.7,
}

boundary_config = {
    **beta_plots.BOUNDARY_BASE_CONFIG,
    **beta_plots.BOUNDARY_DEFAULT_CONFIG,
    **beta_plots.BOUNDARY_CONFIG_PRESETS.get(func, {}),
}

positions = get_regime_positions(func)
regime_overlay = build_regime_overlay_from_shape(train_loss_grid.shape, pos_frac=positions)
boundary_overlays, combined_overlay, fused_overlay, mapping_info = build_boundary_overlays(
    plot_data,
    config=boundary_config,
    metric_keys=('training_loss', 'test_error'),
)
if mapping_info:
    print(f"Fused mapping: train_idx={mapping_info.get('train_indices')} -> test_idx={mapping_info.get('test_indices')}")

# Per-metric plotting configs (Hessian skipped because data is missing)
metrics = [
    ('test_error', 'Test Error', {'vmin': 0.15, 'vmax': 0.7}),
    ('training_loss', 'Training Error', {'vmin': 0.06, 'vmax': 0.3}),
]

for metric, metric_title, cfg in metrics:
    overlay = combined_overlay if metric in ('training_loss', 'test_error') else boundary_overlays.get(metric)
    out_name = f"{metric}_ad_scale_Jan20_AD.pdf"
    out_path = REPO_ROOT / 'plots' / 'pdf' / out_name
    plot_2d_phase(
        plot_data[metric],
        x_axis,
        y_axis,
        metric,
        metric_title,
        func,
        use_data_range=cfg.get('use_data_range', True),
        log_scale=cfg.get('log_scale', False),
        regime_overlay=merge_overlays(regime_overlay, overlay),
        boundary_data=plot_data,
        x_label_override=r"$K$",
        y_label_override='Number of Samples',
        save_path=str(out_path),
        vmin_override=cfg.get('vmin'),
        vmax_override=cfg.get('vmax'),
    )

print('Updated plots saved to plots/pdf/.')


Fused mapping: train_idx=[0 1 2 3 4 5 6] -> test_idx=[0 1 2 3 4 4 3]
Updated plots saved to plots/pdf/.
